# 00 Collect

Turns the run directories under runs/MAIN into the tables every other notebook reads: results/MAIN.csv (one row per run, made by mmb.analysis.collect) and results/validity_MAIN.csv (validity proxies of the crossing inputs, made by mmb.validity.score). No logic is copied here; the notebook calls the same functions as scripts/analysis/export_to_paper.sh.

In [ ]:
from pathlib import Path
import pandas as pd
from mmb.analysis import collect as col

RUNS = Path("../runs/MAIN") if Path("../runs/MAIN").exists() else Path("runs/MAIN")
RESULTS = Path("../results") if Path("../results").exists() else Path("results")
RESULTS.mkdir(exist_ok=True)
REBUILD = False   # True re-reads every run directory (minutes); False reuses results/MAIN.csv

## Run table

One row per run: arm, task, SUT, cell, seed, whether and when it crossed, the floor, the magnitudes at the crossing, the edit counts and the wall-clock time. The arm is derived from the run configuration, never from the directory name.

In [ ]:
if REBUILD or not (RESULTS / "MAIN.csv").exists():
    runs = col.collect(RUNS, {"exp_id": "MAIN"})
    runs.to_csv(RESULTS / "MAIN.csv", index=False)
runs = pd.read_csv(RESULTS / "MAIN.csv")
runs.groupby(["task", "sut", "arm"]).size().unstack("arm").fillna(0).astype(int)

## Completeness

Runs whose search stopped before the budget are flagged incomplete. They are re-run, not dropped, so this count should reach zero before final numbers.

In [ ]:
print("complete:", int(runs["complete"].sum()), "of", len(runs))
runs.loc[~runs["complete"], ["arm", "task", "sut", "cell_id", "rng_seed", "evals_total"]].head(20)

## Validity proxies

The proxies load CLIP, DINOv2, LPIPS and MiniLM from the local cache and score every crossing input against the codec reconstruction of its seed, so this step is slow and off by default. Set RUN_VALIDITY to recompute results/validity_MAIN.csv.

In [ ]:
RUN_VALIDITY = False
if RUN_VALIDITY:
    from mmb.validity import score
    from mmb.validity.proxies import ProxyModels
    v = score.score_runs(RUNS, ProxyModels(local_files_only=True), {"exp_id": "MAIN"})
    v.to_csv(RESULTS / "validity_MAIN.csv", index=False)
validity = pd.read_csv(RESULTS / "validity_MAIN.csv")
validity.groupby(["view", "arm"]).size().unstack("view")